In [3]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
df = pd.read_csv('../data/insurance_data.csv')
df.head()

from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(df[["age","affordibility"]], df[["bought_insurance"]])

In [5]:
class MiNN:
    def __init__(self):
        self.w1 = 1
        self.w2 = 1
        self.bias = 0
    
    def fit(self, x, y, epochs, loss_threshold):
        self.w1, self.w2, self.bias = self.gradient_decent(x['age'], x['affordibility'], y, epochs, loss_threshold)
        
    def predict(self, x_test):
        weighted_sum = self.w1*x_test['age'] + self.w2*x_test['affordibility'] + self.bias
        return self.sigmoid_numpy(weighted_sum)
        
    def sigmoid_numpy(self, x):
        return 1 / (1 + np.exp(-x))
    
    def log_loss(self, y_true, y_prediction):
        epsilon = 1e-15
        y_true = np.asarray(y_true).reshape(-1)
        y_prediction = np.asarray(y_prediction).reshape(-1)
        y_prediction = np.clip(y_prediction, epsilon, 1 - epsilon)
        return -np.mean(y_true * np.log(y_prediction) + (1 - y_true) * np.log(1 - y_prediction))
    
    def gradient_decent(self, age, affordibility, y_true, epochs, loss_threshold):
        # w1, w2, bias
        w1 = w2 = 1
        bias = 0
        rate = 0.5
        n = len(age)

        # Convert age, affordibility, y_true to numpy arrays at the start for consistent numerical operations
        age_arr = age.to_numpy()
        affordibility_arr = affordibility.to_numpy()
        y_true_arr = y_true.to_numpy().reshape(-1)

        for i in range(epochs):
            weighted_sum = w1*age_arr + w2*affordibility_arr + bias
            y_predicted = self.sigmoid_numpy(weighted_sum) # sigmoid_numpy expects numpy array

            loss = self.log_loss(y_true_arr, y_predicted) # Pass numpy array y_true_arr to log_loss

            # np.dot for 1D arrays performs inner product. No need for transpose for 1D arrays.
            # np.dot(1D array, 1D array) -> scalar
            w1d = (1/n) * np.dot(age_arr, (y_predicted - y_true_arr))
            w2d = (1/n) * np.dot(affordibility_arr, (y_predicted - y_true_arr))

            bias_d = np.mean(y_predicted - y_true_arr)

            w1 = w1 - rate * w1d
            w2 = w2 - rate * w2d
            bias = bias - rate * bias_d

            # Use .item() for scalar numpy values to print them as regular numbers
            print(f'Epoch:{i}, w1:{w1.item()}, w2:{w2.item()}, bias:{bias.item()}, loss:{loss.item()}')

            if loss <= loss_threshold:
                break

        return w1, w2, bias

In [6]:
custom_model = MiNN()
custom_model.fit(x_train, y_train, epochs=5, loss_threshold=0.4631)

Epoch:0, w1:-7.238095233006984, w2:0.9047619049019523, bias:-0.26190476163081133, loss:14.26752073046512
Epoch:1, w1:5.28571429080254, w2:1.1190476191876666, bias:-0.02380952353557325, loss:16.447036378528896
Epoch:2, w1:-2.9523809472926974, w2:1.0238095239495713, bias:-0.28571428544033517, loss:18.092158853130936
Epoch:3, w1:9.571428576516826, w2:1.2380952382352854, bias:-0.04761904734509709, loss:16.447036378528896
Epoch:4, w1:1.333333338421589, w2:1.1428571429971903, bias:-0.309523809249859, loss:18.092158853130936


In [7]:
custom_model.predict(x_test)

16    1.0
0     1.0
19    1.0
26    1.0
8     1.0
23    1.0
24    1.0
dtype: float64